# Stanford De-Identifier PHI NER

Model: `StanfordAIMI/stanford-deidentifier-base`

This notebook tests Stanford AIMI's de-identification model on a clinical sentence. It is designed for PHI detection, not medication or symptom extraction.

## Supported Entities

| Entity | Meaning |
| --- | --- |
| `DATE` | Dates and date-like expressions |
| `HCW` | Health care worker names, such as doctors or clinical staff |
| `HOSPITAL` | Hospital or care-site names |
| `ID` | Identifiers |
| `PATIENT` | Patient names |
| `PHONE` | Phone numbers |
| `VENDOR` | Vendor or organization names |
| `O` | Outside any entity |

The notebook returns only detected entity spans, so `O` rows are omitted.

In [1]:
from __future__ import annotations

import pandas as pd
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline

MODEL_NAME = "StanfordAIMI/stanford-deidentifier-base"
device = 0 if torch.cuda.is_available() else -1

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

ner_pipeline = pipeline(
    task="token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=device,
)

print(f"Loaded {MODEL_NAME} on {'cuda' if device == 0 else 'cpu'}.")
print("Labels:")
print(model.config.id2label)

Device set to use cpu


Loaded StanfordAIMI/stanford-deidentifier-base on cpu.
Labels:
{0: 'O', 1: 'VENDOR', 2: 'DATE', 3: 'HCW', 4: 'HOSPITAL', 5: 'ID', 6: 'PATIENT', 7: 'PHONE'}


Edit `input_text`, then run the detection cells.

In [2]:
input_text = "On 03/12/2019, Dr. Emily Smith at St. Mary's Hospital in Stuttgart prescribed ibuprofen 400 mg to 67-year-old Mr. David Jones, patient ID MRN-45892, for severe left knee pain caused by osteoarthritis after an X-ray showed joint-space narrowing."
input_text

"On 03/12/2019, Dr. Emily Smith at St. Mary's Hospital in Stuttgart prescribed ibuprofen 400 mg to 67-year-old Mr. David Jones, patient ID MRN-45892, for severe left knee pain caused by osteoarthritis after an X-ray showed joint-space narrowing."

In [3]:
def detect_phi_entities(text: str) -> pd.DataFrame:
    predictions = ner_pipeline(text)
    rows = []

    for prediction in predictions:
        rows.append(
            {
                "entity": prediction.get("entity_group", prediction.get("entity")),
                "text": text[prediction["start"]:prediction["end"]],
                "pipeline_text": prediction["word"],
                "start": prediction["start"],
                "end": prediction["end"],
                "score": round(float(prediction["score"]), 4),
            }
        )

    return pd.DataFrame(
        rows,
        columns=["entity", "text", "pipeline_text", "start", "end", "score"],
    )


entities = detect_phi_entities(input_text)
entities

,entity,text,pipeline_text,start,end,score
0,DATE,03/12/2019,03 / 12 / 2019,3,13,0.9981
1,HCW,Dr. Emily Smith,dr. emily smith,15,30,0.9981
2,HOSPITAL,St. Mary's Hospital,st. mary ' s hospital,34,53,0.9974
3,HOSPITAL,Stuttgart,stuttgart,57,66,0.9996
4,PATIENT,. David Jones,. david jones,112,125,0.7544
5,ID,MRN-45892,mrn - 45892,138,147,0.9127


In [4]:
if entities.empty:
    print("No PHI entities detected.")
else:
    for row in entities.itertuples(index=False):
        print(f"{row.entity}: {row.text} ({row.score})")

DATE: 03/12/2019 (0.9981)
HCW: Dr. Emily Smith (0.9981)
HOSPITAL: St. Mary's Hospital (0.9974)
HOSPITAL: Stuttgart (0.9996)
PATIENT: . David Jones (0.7544)
ID: MRN-45892 (0.9127)


Expected behavior on the default sentence: this model may detect the date, doctor/staff name, hospital name, and patient name. It should not detect `ibuprofen`, `knee`, or `pain` because those are not part of this model's PHI label space.